# Perform data loading or upsert from CSV files to Google Cloud SQL tables

In [1]:
# import packages that will be used

import io
import os
from typing import Dict, List, Sequence
from zoneinfo import ZoneInfo

import pandas as pd
from dotenv import load_dotenv
from google.cloud import storage
from google.cloud.sql.connector import Connector
from sqlalchemy import create_engine, text
from pathlib import Path


E0000 00:00:1781070462.682533   87498 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1781070462.682615   87498 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_rejected' registered more than once. Ignoring later registration.
E0000 00:00:1781070462.682621   87498 instrument.cc:563] Metric with name 'grpc.resource_quota.connections_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1781070462.682624   87498 instrument.cc:563] Metric with name 'grpc.resource_quota.instantaneous_memory_pressure' registered more than once. Ignoring later registration.
E0000 00:00:1781070462.682626   87498 instrument.cc:563] Metric with name 'grpc.resource_quota.memory_pressure_control_value' registered more than once. Ignoring later registration.


In [2]:
# ------------------------------------------------------------
# Load environment variables
# ------------------------------------------------------------
env_path = Path.cwd().parent / "src/.env"
if not env_path.exists():
    raise FileNotFoundError(f".env file not found: {env_path}")

load_dotenv(dotenv_path=env_path, override=True)

INSTANCE_CONNECTION_NAME = os.getenv("INSTANCE_CONNECTION_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASS")
DB_NAME = os.getenv("DB_NAME")

GCS_BUCKET_NAME = os.getenv("GCS_BUCKET_NAME")
GCS_PREFIX = os.getenv("GCS_PREFIX")
BATCH_TIME = os.getenv("BATCH_TIME")  # Optional. Example: 2026-06-08T00:30:00+08:00 or 2026-06-08 00:30:00
BATCH_TIMEZONE = os.getenv("BATCH_TIMEZONE", "Asia/Singapore")  # Use SGT by default
POSTGRES_SCHEMA = os.getenv("POSTGRES_SCHEMA")

# LOAD_MODE controls table writing:
# - append: insert rows only
# - replace: drop/recreate table using pandas to_sql
# - upsert: INSERT ... ON CONFLICT DO UPDATE using table primary/composite key
LOAD_MODE = os.getenv("LOAD_MODE", "append").lower()

required_vars = {
    "INSTANCE_CONNECTION_NAME": INSTANCE_CONNECTION_NAME,
    "DB_USER": DB_USER,
    "DB_PASS": DB_PASS,
    "DB_NAME": DB_NAME,
    "GCS_BUCKET_NAME": GCS_BUCKET_NAME,
    "GCS_PREFIX": GCS_PREFIX,
    "BATCH_TIME": BATCH_TIME,
    "BATCH_TIMEZONE": BATCH_TIMEZONE,
    "POSTGRES_SCHEMA": POSTGRES_SCHEMA,
}

missing_vars = [key for key, value in required_vars.items() if not value]

if missing_vars:
    raise ValueError(f"Missing required .env values: {missing_vars}")
else:
    print("All required environment variables are set.\n")

if LOAD_MODE not in {"append", "replace", "upsert"}:
    raise ValueError("LOAD_MODE must be one of: append, replace, upsert")

print(f"GCP_PROJECT_ID: {os.getenv('GCP_PROJECT_ID')}")
print(f"Using Cloud SQL instance: {INSTANCE_CONNECTION_NAME}")
print(f"Using GCS bucket: {GCS_BUCKET_NAME} with prefix: {GCS_PREFIX}")
print(f"Database user: {DB_USER}")
print(f"Database name: {DB_NAME}")
print(f"Database password: {'*' * len(DB_PASS)}")  # Mask the password in logs
print(f"Using PostgreSQL schema: {POSTGRES_SCHEMA}")
print(f"Using load mode: {LOAD_MODE}")
print(f"Using batch timezone: {BATCH_TIMEZONE}")
print(f"Using batch time: {BATCH_TIME}")

All required environment variables are set.

GCP_PROJECT_ID: sctp-team2-project2-elt
Using Cloud SQL instance: sctp-team2-project2-elt:us-central1:sctp-m2-olist
Using GCS bucket: m2_olin_raw with prefix: INCR 2026-06-01/
Database user: postgres
Database name: olist
Database password: ****************
Using PostgreSQL schema: oltp
Using load mode: upsert
Using batch timezone: Asia/Singapore
Using batch time: 2026-06-10T00:30:00+08:00


## Upsert support

This notebook supports three `LOAD_MODE` values from `.env`:

- `append`: insert rows only
- `replace`: drop and recreate the target table using pandas `to_sql`
- `upsert`: use PostgreSQL `INSERT ... ON CONFLICT DO UPDATE`

For `upsert`, each table must have a matching primary key or unique constraint in PostgreSQL. `olist_geolocation` has no natural primary key, so it falls back to append mode.


In [ ]:
# ------------------------------------------------------------
# CSV file to PostgreSQL table mapping
# ------------------------------------------------------------

CSV_TO_TABLE: Dict[str, str] = {
    "olist_customers_dataset.csv": "olist_customers",
    # "olist_geolocation_dataset.csv": "olist_geolocation",
    "olist_order_items_dataset.csv": "olist_order_items",
    "olist_order_payments_dataset.csv": "olist_order_payments",
    "olist_order_reviews_dataset.csv": "olist_order_reviews",
    "olist_orders_dataset.csv": "olist_orders",
    "olist_products_dataset.csv": "olist_products",
    "olist_sellers_dataset.csv": "olist_sellers",
    "product_category_name_translation.csv": "product_category_name_translation",
}

# PostgreSQL conflict keys used only when LOAD_MODE="upsert".
# olist_geolocation intentionally has no key because the Olist geolocation source
# does not have a reliable natural primary key. It will fall back to append mode.
UPSERT_KEYS: Dict[str, List[str]] = {
    "olist_customers": ["customer_id"],
    "olist_orders": ["order_id"],
    "olist_order_items": ["order_id", "order_item_id"],
    "olist_order_payments": ["order_id", "payment_sequential"],
    "olist_order_reviews": ["review_id", "order_id"],
    "olist_products": ["product_id"],
    "olist_sellers": ["seller_id"],
    "product_category_name_translation": ["product_category_name"],
}


# ------------------------------------------------------------
# Cloud SQL PostgreSQL connection
# ------------------------------------------------------------
connector = Connector()


def getconn():
    return connector.connect(
        INSTANCE_CONNECTION_NAME,
        "pg8000",
        user=DB_USER,
        password=DB_PASS,
        db=DB_NAME,
    )


engine = create_engine(
    "postgresql+pg8000://",
    creator=getconn,
)


# ------------------------------------------------------------
# Create schema if it does not exist
# ------------------------------------------------------------
def create_schema_if_not_exists() -> None:
    with engine.begin() as conn:
        conn.execute(text(f'CREATE SCHEMA IF NOT EXISTS "{POSTGRES_SCHEMA}"'))
    print(f"Schema ready: {POSTGRES_SCHEMA}")


def quote_ident(identifier: str) -> str:
    """Safely quote a PostgreSQL identifier such as a schema, table, or column name."""
    return '"' + identifier.replace('"', '""') + '"'


def chunk_records(records: List[dict], chunk_size: int = 1_000):
    """Yield records in chunks to avoid overly large INSERT statements."""
    for start in range(0, len(records), chunk_size):
        yield records[start:start + chunk_size]


def upsert_dataframe_to_postgres(
    df: pd.DataFrame,
    table_name: str,
    key_columns: Sequence[str],
    chunk_size: int = 1_000,
) -> None:
    """
    Upsert a DataFrame into PostgreSQL using INSERT ... ON CONFLICT DO UPDATE.

    Requirements:
    - The target table must already exist.
    - key_columns must match a PRIMARY KEY or UNIQUE constraint in PostgreSQL.
    - This is intended for incremental loads where existing rows should be updated.
    """
    if not key_columns:
        raise ValueError(f"No upsert key defined for table: {table_name}")

    missing_key_cols = [col for col in key_columns if col not in df.columns]
    if missing_key_cols:
        raise ValueError(f"Missing upsert key columns in DataFrame for {table_name}: {missing_key_cols}")

    columns = list(df.columns)

    full_table_name = f"{quote_ident(POSTGRES_SCHEMA)}.{quote_ident(table_name)}"
    insert_columns = ", ".join(quote_ident(col) for col in columns)
    values_columns = ", ".join(f":{col}" for col in columns)
    conflict_columns = ", ".join(quote_ident(col) for col in key_columns)

    # Preserve created_at from the original row. Update the other non-key columns.
    update_columns = [
        col for col in columns
        if col not in key_columns and col != "created_at"
    ]

    if update_columns:
        update_set = ",\n            ".join(
            f"{quote_ident(col)} = EXCLUDED.{quote_ident(col)}"
            for col in update_columns
        )
        conflict_action = f"DO UPDATE SET\n            {update_set}"
    else:
        conflict_action = "DO NOTHING"

    sql = text(f"""
        INSERT INTO {full_table_name} ({insert_columns})
        VALUES ({values_columns})
        ON CONFLICT ({conflict_columns})
        {conflict_action}
    """)

    records = df.where(pd.notnull(df), None).to_dict(orient="records")

    with engine.begin() as conn:
        for record_chunk in chunk_records(records, chunk_size):
            conn.execute(sql, record_chunk)


def write_dataframe_to_postgres(
    df: pd.DataFrame,
    table_name: str,
    load_mode: str,
) -> None:
    """
    Write a DataFrame into PostgreSQL using append, replace, or upsert.
    """
    if load_mode in {"append", "replace"}:
        df.to_sql(
            table_name,
            engine,
            schema=POSTGRES_SCHEMA,
            if_exists=load_mode,
            index=False,
            chunksize=1_000,
            method="multi",
        )
        return

    if load_mode == "upsert":
        key_columns = UPSERT_KEYS.get(table_name)

        if not key_columns:
            print(
                f"Table {POSTGRES_SCHEMA}.{table_name} has no upsert key configured. "
                "Falling back to append mode."
            )
            df.to_sql(
                table_name,
                engine,
                schema=POSTGRES_SCHEMA,
                if_exists="append",
                index=False,
                chunksize=1_000,
                method="multi",
            )
            return

        upsert_dataframe_to_postgres(
            df=df,
            table_name=table_name,
            key_columns=key_columns,
            chunk_size=1_000,
        )
        return

    raise ValueError("load_mode must be one of: append, replace, upsert")


# ------------------------------------------------------------
# Batch timestamp helper
# ------------------------------------------------------------
def get_batch_timestamp() -> pd.Timestamp:
    """
    Return a timezone-aware batch timestamp.

    Defaults to Singapore time (Asia/Singapore). If BATCH_TIME is provided:
    - If it has an explicit timezone offset, convert it to BATCH_TIMEZONE.
    - If it is timezone-naive, localize it to BATCH_TIMEZONE.

    Note: PostgreSQL TIMESTAMPTZ stores an absolute instant. Depending on the
    session timezone, it may display as UTC or SGT, but the instant is the same.
    """
    batch_tz = ZoneInfo(BATCH_TIMEZONE)

    if BATCH_TIME:
        ts = pd.Timestamp(BATCH_TIME)
        if ts.tzinfo is None:
            ts = ts.tz_localize(batch_tz)
        else:
            ts = ts.tz_convert(batch_tz)
    else:
        ts = pd.Timestamp.now(tz=batch_tz)

    return ts


# ------------------------------------------------------------
# Load one CSV from GCS into PostgreSQL
# ------------------------------------------------------------
def load_csv_from_gcs_to_postgres(
    storage_client: storage.Client,
    filename: str,
    table_name: str,
    load_mode: str = "append",
) -> None:
    bucket = storage_client.bucket(GCS_BUCKET_NAME)
    blob_path = f"{GCS_PREFIX.rstrip('/')}/{filename}"
    blob = bucket.blob(blob_path)

    if not blob.exists():
        raise FileNotFoundError(f"File not found: gs://{GCS_BUCKET_NAME}/{blob_path}")

    gsfilepath = f"gs://{GCS_BUCKET_NAME}/{blob_path}"
    print(f"\nReading {gsfilepath}")

    csv_bytes = blob.download_as_bytes()
    df = pd.read_csv(io.BytesIO(csv_bytes))

    # Add ingestion metadata. Use BATCH_TIME when provided for repeatable tests,
    # otherwise use the current timestamp in BATCH_TIMEZONE. Default is SGT.
    batch_ts = get_batch_timestamp()

    print(f"Current/Batch time ({BATCH_TIMEZONE}): {batch_ts}")
    print(f"Equivalent UTC time: {batch_ts.tz_convert('UTC')}")

    df["created_at"] = batch_ts
    df["updated_at"] = batch_ts
    df["is_deleted"] = False
    df["source_file"] = filename
    df["source_gcs_path"] = gsfilepath
    df["batch_name"] = GCS_PREFIX.strip("/").split("/")[-1]

    print(f"Loading {len(df):,} rows into {POSTGRES_SCHEMA}.{table_name} using load_mode={load_mode}")

    write_dataframe_to_postgres(
        df=df,
        table_name=table_name,
        load_mode=load_mode,
    )

    print(f"Loaded {POSTGRES_SCHEMA}.{table_name}")

    with engine.begin() as conn:
        print(f"Analyzing {POSTGRES_SCHEMA}.{table_name} for query performance...")
        conn.execute(text(f'ANALYZE "{POSTGRES_SCHEMA}"."{table_name}"'))

        table_stats_sql = text("""
            SELECT
                schemaname,
                relname AS table_name,
                n_live_tup AS estimated_live_rows,
                n_dead_tup AS estimated_dead_rows,
                last_analyze,
                last_autoanalyze
            FROM pg_stat_user_tables
            WHERE schemaname = :schema_name
              AND relname = :table_name
        """)

        table_stats = pd.read_sql(
            table_stats_sql,
            conn,
            params={
                "schema_name": POSTGRES_SCHEMA,
                "table_name": table_name,
            },
        )

        print("\nPostgreSQL table statistics:")
        print(table_stats)


# ------------------------------------------------------------
# Main
# ------------------------------------------------------------
def main() -> None:
    create_schema_if_not_exists()

    storage_client = storage.Client()

    for filename, table_name in CSV_TO_TABLE.items():
        print(f"\nProcessing file: {filename} -> table: {table_name}")
        load_csv_from_gcs_to_postgres(
            storage_client=storage_client,
            filename=filename,
            table_name=table_name,
            load_mode=LOAD_MODE,
        )

    connector.close()
    print("\nAll CSV files loaded successfully.")


if __name__ == "__main__":
    main()


Schema ready: oltp

Processing file: product_category_name_translation.csv -> table: product_category_name_translation

Reading gs://m2_olin_raw_johnphs/Initial Load/product_category_name_translation.csv
Current UTC time: 2026-06-04 18:07:37.499749+00:00
Loading 71 rows into oltp.product_category_name_translation
Loaded oltp.product_category_name_translation
Analyzing oltp.product_category_name_translation for query performance...

PostgreSQL table statistics:
  schemaname                         table_name  estimated_live_rows  \
0       oltp  product_category_name_translation                   71   

   estimated_dead_rows                     last_analyze last_autoanalyze  
0                    0 2026-06-04 18:07:52.098399+00:00             None  

All CSV files loaded successfully.
